# Notebook 04 - Modelisation
Comparaison de quatre familles de modeles et quatre strategies de gestion du
desequilibre avec validation croisee stratifiee a cinq splits.

Pour une comparaison equitable, les modeles utilises avec SMOTE ou
undersampling ne conservent pas simultanement les poids de classes.


In [1]:
import sys
from pathlib import Path
import os

# Determine the project root dynamically
# This assumes the notebook is either in the project root or in a 'notebooks' subdirectory.
current_path = Path(os.getcwd()).resolve()
if current_path.name == 'notebooks':
    ROOT = current_path.parent
else:
    # Assume current_path is the project root
    ROOT = current_path
sys.path.insert(0, str(ROOT))

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display
from imblearn.combine import SMOTETomek
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.under_sampling import RandomUnderSampler
from sklearn.base import clone
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.tree import DecisionTreeClassifier
from xgboost import XGBClassifier


In [2]:
train = pd.read_csv(ROOT / 'data' / 'processed' / 'train.csv')
X_train, y_train = train.drop(columns='bad_nutrition'), train['bad_nutrition'].astype(int)
preprocessor = joblib.load(ROOT / 'models' / 'preprocessor.joblib')
class_ratio = float((y_train == 0).sum() / (y_train == 1).sum())

weighted_models = {
    'LogisticRegression': LogisticRegression(max_iter=5000, class_weight='balanced', random_state=42),
    'DecisionTree': DecisionTreeClassifier(class_weight='balanced', random_state=42),
    'RandomForest': RandomForestClassifier(n_estimators=200, class_weight='balanced_subsample', n_jobs=1, random_state=42),
    'XGBoost': XGBClassifier(n_estimators=200, max_depth=5, learning_rate=0.1, scale_pos_weight=class_ratio, n_jobs=1, random_state=42, eval_metric='logloss'),
}
unweighted_models = {
    'LogisticRegression': LogisticRegression(max_iter=5000, random_state=42),
    'DecisionTree': DecisionTreeClassifier(random_state=42),
    'RandomForest': RandomForestClassifier(n_estimators=200, n_jobs=1, random_state=42),
    'XGBoost': XGBClassifier(n_estimators=200, max_depth=5, learning_rate=0.1, n_jobs=1, random_state=42, eval_metric='logloss'),
}
samplers = {
    'baseline': None,
    'smote': SMOTE(random_state=42),
    'undersample': RandomUnderSampler(random_state=42),
    'smote_tomek': SMOTETomek(random_state=42),
}
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

rows = []
for model_name in weighted_models:
    for strategy_name, sampler in samplers.items():
        classifier = weighted_models[model_name] if sampler is None else unweighted_models[model_name]
        steps = [('preprocessor', clone(preprocessor))]
        if sampler is not None:
            steps.append(('sampler', sampler))
        steps.append(('clf', classifier))
        pipeline = ImbPipeline(steps)
        scores = cross_val_score(pipeline, X_train, y_train, scoring='f1', cv=cv, n_jobs=1)
        rows.append({
            'model': model_name,
            'strategy': strategy_name,
            'mean_f1': scores.mean(),
            'std_f1': scores.std(),
        })
        print(f'{model_name:20s} {strategy_name:12s} F1={scores.mean():.4f} +/- {scores.std():.4f}')

results = pd.DataFrame(rows).sort_values('mean_f1', ascending=False).reset_index(drop=True)
results.to_csv(ROOT / 'models' / 'model_selection_results.csv', index=False)
assert results['model'].nunique() >= 4
assert results['strategy'].nunique() >= 4
display(results)


LogisticRegression   baseline     F1=0.7410 +/- 0.0185
LogisticRegression   smote        F1=0.7429 +/- 0.0152
LogisticRegression   undersample  F1=0.7332 +/- 0.0127
LogisticRegression   smote_tomek  F1=0.7427 +/- 0.0155
DecisionTree         baseline     F1=0.8360 +/- 0.0192
DecisionTree         smote        F1=0.8366 +/- 0.0092
DecisionTree         undersample  F1=0.7875 +/- 0.0203
DecisionTree         smote_tomek  F1=0.8372 +/- 0.0091
RandomForest         baseline     F1=0.8652 +/- 0.0103
RandomForest         smote        F1=0.8813 +/- 0.0156
RandomForest         undersample  F1=0.8347 +/- 0.0218
RandomForest         smote_tomek  F1=0.8813 +/- 0.0177
XGBoost              baseline     F1=0.8868 +/- 0.0239
XGBoost              smote        F1=0.8814 +/- 0.0130
XGBoost              undersample  F1=0.8452 +/- 0.0210
XGBoost              smote_tomek  F1=0.8797 +/- 0.0187


,model,strategy,mean_f1,std_f1
0,XGBoost,baseline,0.886777,0.023931
1,XGBoost,smote,0.881411,0.013035
2,RandomForest,smote,0.881306,0.015601
3,RandomForest,smote_tomek,0.881263,0.017665
4,XGBoost,smote_tomek,0.879745,0.018717
5,RandomForest,baseline,0.865216,0.010263
6,XGBoost,undersample,0.845152,0.020982
7,DecisionTree,smote_tomek,0.837179,0.009080
8,DecisionTree,smote,0.836627,0.009188
9,DecisionTree,baseline,0.835993,0.019219


In [3]:
comparison = results.assign(
    score=results.apply(lambda row: f"{row.mean_f1:.4f} +/- {row.std_f1:.4f}", axis=1)
).pivot(index='model', columns='strategy', values='score')
display(comparison)
best = results.iloc[0]
print(f"Meilleure combinaison: {best.model} + {best.strategy}, F1 CV={best.mean_f1:.4f} +/- {best.std_f1:.4f}")


strategy,baseline,smote,smote_tomek,undersample
model,,,,
DecisionTree,0.8360 +/- 0.0192,0.8366 +/- 0.0092,0.8372 +/- 0.0091,0.7875 +/- 0.0203
LogisticRegression,0.7410 +/- 0.0185,0.7429 +/- 0.0152,0.7427 +/- 0.0155,0.7332 +/- 0.0127
RandomForest,0.8652 +/- 0.0103,0.8813 +/- 0.0156,0.8813 +/- 0.0177,0.8347 +/- 0.0218
XGBoost,0.8868 +/- 0.0239,0.8814 +/- 0.0130,0.8797 +/- 0.0187,0.8452 +/- 0.0210


Meilleure combinaison: XGBoost + baseline, F1 CV=0.8868 +/- 0.0239


## Synthese
Le meilleur couple est selectionne exclusivement sur le jeu d'entrainement via
validation croisee. Le jeu de test reste intact jusqu'a l'evaluation finale.
